In [5]:
import logging
from typing import List, Dict
from datasets import load_dataset

log = logging.getLogger(__name__)

class MathDatasetBuilder:
    def __init__(self, dataset_name: str, max_prompt_len: int = 512):
        self.dataset_name = dataset_name
        self.max_prompt_len = max_prompt_len
        self.system_prompt = (
            "You are a mathematics expert. "
            "Solve the problem step by step and enclose your final answer in \\boxed{}."
        )

    def format_chat_prompt(self, problem: str, tokenizer) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": problem},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def load_train_data(self, tokenizer) -> List[Dict]:
        log.info(f"Loading dataset: {self.dataset_name}")
        ds = load_dataset(self.dataset_name, split="train") 
        out = []
        for item in ds:
            problem = item.get("problem", "")
            if not problem: continue
                
            txt = self.format_chat_prompt(problem, tokenizer)
            ids = tokenizer.encode(txt, add_special_tokens=False)
            if len(ids) <= self.max_prompt_len:
                out.append({"prompt": txt, "prompt_ids": ids})
                
        log.info(f"Loaded {len(out)} valid training examples.")
        return out

In [6]:
import logging
import torch
from transformers import AutoModelForCausalLM

log = logging.getLogger(__name__)

def load_policy_and_ref_models(model_name: str, dtype: torch.dtype, device_map: str = "auto"):
    """Loads the trainable Actor (Policy) and frozen Reference models."""
    
    log.info(f"Loading Actor model (Policy): {model_name}")
    policy_model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=dtype, 
        device_map=device_map, 
        trust_remote_code=True
    )
    policy_model.gradient_checkpointing_enable()
    policy_model.train()

    log.info(f"Loading Reference model (Frozen): {model_name}")
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=dtype, 
        device_map=device_map, 
        trust_remote_code=True
    )
    ref_model.eval()
    
    # Đóng băng toàn bộ trọng số của Reference Model
    for param in ref_model.parameters(): 
        param.requires_grad_(False)

    return policy_model, ref_model

In [ ]:
import torch
import logging
from transformers import AutoTokenizer, AutoModel

log = logging.getLogger(__name__)

class RewardModelScorer:
    def __init__(self, rm_name: str, dtype: torch.dtype, device: str):
        self.device = device
        log.info(f"Initializing Neural Reward Model: {rm_name}")
        
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            rm_name, 
            trust_remote_code=True
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.model = AutoModel.from_pretrained(
            rm_name,
            torch_dtype=dtype,
            trust_remote_code=True,
            device_map=device
        )
        self.model.eval()

    def get_reward(self, questions: list[str], responses: list[str]) -> torch.Tensor:

        rewards = []
        for q, r in zip(questions, responses):
            messages = [
                {"role": "user", "content": q},
                {"role": "assistant", "content": r}
            ]
            
            # Format
            text = self.tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=False
            )
            
            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                
                if hasattr(outputs, "score"):
                    score = outputs.score[0].item()
                elif hasattr(outputs, "logits"):
                    score = outputs.logits[0, 0].item()
                else:
                    # Fallback
                    score = outputs[0][0].item()
                    
            rewards.append(score)
            
        return torch.tensor(rewards, dtype=torch.float32, device=self.device)

In [ ]:
import numpy as np
from typing import List

def compute_lpro_advantages(
    rewards: List[float], 
    lengths: List[int], 
    lambda_len: float = 0.10, 
    eps: float = 1e-8
) -> np.ndarray:

    r = np.array(rewards, dtype=np.float64)
    L = np.array(lengths, dtype=np.float64)

    # Z-score chuẩn hóa
    z_r = (r - r.mean()) / (r.std() + eps)
    z_L = (L - L.mean()) / (L.std() + eps)

    advantages = z_r - lambda_len * z_L
    return advantages

In [ ]:
import torch
from typing import Tuple

def compute_dapo_token_loss(
    new_logprobs: torch.Tensor, 
    old_logprobs: torch.Tensor, 
    advantage: float, 
    mask: torch.Tensor, 
    eps_low: float = 0.20, 
    eps_high: float = 0.28
) -> Tuple[torch.Tensor, int]:

    ratio = torch.exp(new_logprobs - old_logprobs)
    ratio_clipped = torch.clamp(ratio, 1.0 - eps_low, 1.0 + eps_high)

    adv_tensor = torch.full_like(new_logprobs, advantage)
    surrogate1 = ratio * adv_tensor
    surrogate2 = ratio_clipped * adv_tensor

    # Pessimistic bound (min) và đổi dấu vì PyTorch dùng Gradient Descent (minimize)
    token_losses = -torch.min(surrogate1, surrogate2)
    
    # Chỉ tính loss trên các token không phải padding
    masked_loss = token_losses * mask
    
    return masked_loss.sum(), mask.sum().item()

In [ ]:
import os
import math
import random
import logging
import torch
import numpy as np
from torch.optim import AdamW
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

from nexus.models.policy import load_policy_and_ref_models
from nexus.models.reward import RewardModelScorer
from nexus.rl.advantages import compute_lpro_advantages
from nexus.rl.loss import compute_dapo_token_loss
from nexus.data.builder import MathDatasetBuilder

log = logging.getLogger(__name__)

class NexusTrainer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.bfloat16 if cfg.bf16 and torch.cuda.is_bf16_supported() else torch.float32
        
        self._set_seed()
        self._setup_components()

    def _set_seed(self):
        random.seed(self.cfg.seed)
        np.random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)

    def _setup_components(self):
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_name, trust_remote_code=True)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Models
        self.model, self.ref_model = load_policy_and_ref_models(self.cfg.model_name, self.dtype)
        self.rm_scorer = RewardModelScorer(self.cfg.rm_model_name, self.dtype, self.device)

    @staticmethod
    def get_resp_log_probs(model, full_ids: torch.Tensor, prompt_len: int, no_grad: bool = False) -> torch.Tensor:
        """Hàm helper để lấy log probabilities của token sinh ra."""
        ctx = torch.no_grad() if no_grad else torch.enable_grad()
        with ctx:
            out = model(input_ids=full_ids)
            lp = torch.log_softmax(out.logits[0], dim=-1)
            resp_ids = full_ids[0, prompt_len:]
            resp_lp = lp[prompt_len - 1 : full_ids.shape[1] - 1]
            return resp_lp.gather(1, resp_ids.unsqueeze(-1)).squeeze(-1)

    def train(self):
        # Prepare Data & Optimizer
        dataset_builder = MathDatasetBuilder(self.cfg.dataset_name, self.cfg.max_prompt_len)
        dataset = dataset_builder.load_train_data(self.tokenizer)

        optimizer = AdamW(self.model.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        total_steps = self.cfg.num_epochs * math.ceil(len(dataset) / self.cfg.grad_accum)
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(self.cfg.warmup_ratio * total_steps), total_steps)
        
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        global_step, acc_loss, acc_reward = 0, 0.0, 0.0

        # Training Loop
        for epoch in range(self.cfg.num_epochs):
            random.shuffle(dataset)
            log.info(f"========== EPOCH {epoch + 1}/{self.cfg.num_epochs} ==========")

            for idx, example in enumerate(dataset):
                prompt_ids = example["prompt_ids"]
                prompt_txt = example["prompt"]
                prompt_len = len(prompt_ids)
                prompt_t = torch.tensor([prompt_ids], dtype=torch.long, device=self.device)

                # Generate Responses
                self.model.eval()
                with torch.no_grad():
                    gen_out = self.model.generate(
                        prompt_t.expand(self.cfg.G, -1),
                        max_new_tokens=self.cfg.max_new_tokens,
                        temperature=self.cfg.temperature,
                        do_sample=True,
                        pad_token_id=self.tokenizer.eos_token_id,
                    )
                self.model.train()

                # Parse Lengths & Clean Texts
                lengths, masks, full_seqs, resp_texts = [], [], [], []
                for i in range(self.cfg.G):
                    resp_ids = gen_out[i, prompt_len:]
                    pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
                    actual_len = max(pad_mask.sum().item(), 1)
                    
                    lengths.append(actual_len)
                    masks.append(pad_mask.to(self.device))
                    full_seqs.append(gen_out[i])
                    resp_texts.append(self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True))

                # Reward & LPRO Advantages
                rewards = self.rm_scorer.get_scores([prompt_txt]*self.cfg.G, resp_texts)
                if len(set(rewards)) <= 1: continue # Skip if all rewards are the same
                
                advs = compute_lpro_advantages(rewards, lengths, self.cfg.lambda_len)

                # Loss Computation
                total_loss_sum = torch.tensor(0.0, device=self.device)
                total_n_tokens = 0

                for i in range(self.cfg.G):
                    seq = full_seqs[i].unsqueeze(0).to(self.device)[:, :prompt_len + lengths[i]]
                    mask_i = masks[i][:lengths[i]]

                    with torch.cuda.amp.autocast(enabled=self.cfg.bf16):
                        new_lp = self.get_resp_log_probs(self.model, seq, prompt_len, no_grad=False)
                    old_lp = self.get_resp_log_probs(self.ref_model, seq, prompt_len, no_grad=True)

                    loss_sum, n_valid = compute_dapo_token_loss(
                        new_lp, old_lp.detach(), float(advs[i]), mask_i, self.cfg.eps_low, self.cfg.eps_high
                    )
                    total_loss_sum += loss_sum
                    total_n_tokens += n_valid

                if total_n_tokens == 0: continue

                # Token-level Normalization
                loss = total_loss_sum / total_n_tokens / self.cfg.grad_accum
                loss.backward()

                acc_loss += loss.item() * self.cfg.grad_accum
                acc_reward += float(np.mean(rewards))

                # Gradient Update
                if (idx + 1) % self.cfg.grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                    if global_step % self.cfg.log_steps == 0:
                        log.info(f"Step {global_step} | Loss: {acc_loss/self.cfg.log_steps:.4f} | RM Score: {acc_reward/(self.cfg.log_steps*self.cfg.grad_accum):.3f}")
                        acc_loss, acc_reward = 0.0, 0.0

                    if global_step % self.cfg.save_steps == 0:
                        self.model.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))
                        self.tokenizer.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))

        log.info("Training complete. Saving final model...")
        self.model.save_pretrained(self.cfg.output_dir)
        self.tokenizer.save_pretrained(self.cfg.output_dir)import os
import math
import random
import logging
import torch
import numpy as np
from torch.optim import AdamW
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

from nexus.models.policy import load_policy_and_ref_models
from nexus.models.reward import RewardModelScorer
from nexus.rl.advantages import compute_lpro_advantages
from nexus.rl.loss import compute_dapo_token_loss
from nexus.data.builder import MathDatasetBuilder

log = logging.getLogger(__name__)

class NexusTrainer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.bfloat16 if cfg.bf16 and torch.cuda.is_bf16_supported() else torch.float32
        
        self._set_seed()
        self._setup_components()

    def _set_seed(self):
        random.seed(self.cfg.seed)
        np.random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)

    def _setup_components(self):
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_name, trust_remote_code=True)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Models
        self.model, self.ref_model = load_policy_and_ref_models(self.cfg.model_name, self.dtype)
        self.rm_scorer = RewardModelScorer(self.cfg.rm_model_name, self.dtype, self.device)

    @staticmethod
    def get_resp_log_probs(model, full_ids: torch.Tensor, prompt_len: int, no_grad: bool = False) -> torch.Tensor:
        """Hàm helper để lấy log probabilities của token sinh ra."""
        ctx = torch.no_grad() if no_grad else torch.enable_grad()
        with ctx:
            out = model(input_ids=full_ids)
            lp = torch.log_softmax(out.logits[0], dim=-1)
            resp_ids = full_ids[0, prompt_len:]
            resp_lp = lp[prompt_len - 1 : full_ids.shape[1] - 1]
            return resp_lp.gather(1, resp_ids.unsqueeze(-1)).squeeze(-1)

    def train(self):
        # Prepare Data & Optimizer
        dataset_builder = MathDatasetBuilder(self.cfg.dataset_name, self.cfg.max_prompt_len)
        dataset = dataset_builder.load_train_data(self.tokenizer)

        optimizer = AdamW(self.model.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        total_steps = self.cfg.num_epochs * math.ceil(len(dataset) / self.cfg.grad_accum)
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(self.cfg.warmup_ratio * total_steps), total_steps)
        
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        global_step, acc_loss, acc_reward = 0, 0.0, 0.0

        # Training Loop
        for epoch in range(self.cfg.num_epochs):
            random.shuffle(dataset)
            log.info(f"========== EPOCH {epoch + 1}/{self.cfg.num_epochs} ==========")

            for idx, example in enumerate(dataset):
                prompt_ids = example["prompt_ids"]
                prompt_txt = example["prompt"]
                prompt_len = len(prompt_ids)
                prompt_t = torch.tensor([prompt_ids], dtype=torch.long, device=self.device)

                # Generate Responses
                self.model.eval()
                with torch.no_grad():
                    gen_out = self.model.generate(
                        prompt_t.expand(self.cfg.G, -1),
                        max_new_tokens=self.cfg.max_new_tokens,
                        temperature=self.cfg.temperature,
                        do_sample=True,
                        pad_token_id=self.tokenizer.eos_token_id,
                    )
                self.model.train()

                # Parse Lengths & Clean Texts
                lengths, masks, full_seqs, resp_texts = [], [], [], []
                for i in range(self.cfg.G):
                    resp_ids = gen_out[i, prompt_len:]
                    pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
                    actual_len = max(pad_mask.sum().item(), 1)
                    
                    lengths.append(actual_len)
                    masks.append(pad_mask.to(self.device))
                    full_seqs.append(gen_out[i])
                    resp_texts.append(self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True))

                # Reward & LPRO Advantages
                rewards = self.rm_scorer.get_scores([prompt_txt]*self.cfg.G, resp_texts)
                if len(set(rewards)) <= 1: continue # Skip if all rewards are the same
                
                advs = compute_lpro_advantages(rewards, lengths, self.cfg.lambda_len)

                # Loss Computation
                total_loss_sum = torch.tensor(0.0, device=self.device)
                total_n_tokens = 0

                for i in range(self.cfg.G):
                    seq = full_seqs[i].unsqueeze(0).to(self.device)[:, :prompt_len + lengths[i]]
                    mask_i = masks[i][:lengths[i]]

                    with torch.cuda.amp.autocast(enabled=self.cfg.bf16):
                        new_lp = self.get_resp_log_probs(self.model, seq, prompt_len, no_grad=False)
                    old_lp = self.get_resp_log_probs(self.ref_model, seq, prompt_len, no_grad=True)

                    loss_sum, n_valid = compute_dapo_token_loss(
                        new_lp, old_lp.detach(), float(advs[i]), mask_i, self.cfg.eps_low, self.cfg.eps_high
                    )
                    total_loss_sum += loss_sum
                    total_n_tokens += n_valid

                if total_n_tokens == 0: continue

                # Token-level Normalization
                loss = total_loss_sum / total_n_tokens / self.cfg.grad_accum
                loss.backward()

                acc_loss += loss.item() * self.cfg.grad_accum
                acc_reward += float(np.mean(rewards))

                # Gradient Update
                if (idx + 1) % self.cfg.grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                    if global_step % self.cfg.log_steps == 0:
                        log.info(f"Step {global_step} | Loss: {acc_loss/self.cfg.log_steps:.4f} | RM Score: {acc_reward/(self.cfg.log_steps*self.cfg.grad_accum):.3f}")
                        acc_loss, acc_reward = 0.0, 0.0

                    if global_step % self.cfg.save_steps == 0:
                        self.model.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))
                        self.tokenizer.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))

        log.info("Training complete. Saving final model...")
        self.model.save_pretrained(self.cfg.output_dir)
        self.tokenizer.save_pretrained(self.cfg.output_dir)

In [ ]:
import json
import logging

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

class Config:
    def __init__(self):
        self.model_name      : str   = "Qwen/Qwen2.5-Math-1.5B-Instruct"
        self.rm_model_name   : str   = "Qwen/Qwen2.5-Math-RM-72B"
        self.output_dir      : str   = "./nexus-1.5b"
        self.hub_repo_id     : str   = "YOUR_HF_USERNAME/nexus-1.5b"

        # dataset
        self.dataset_name    : str   = "lighteval/MATH"
        self.max_prompt_len  : int   = 512

        # group sampling
        self.G               : int   = 8      
        self.max_new_tokens  : int   = 1024
        self.temperature     : float = 0.7
        self.top_p           : float = 0.95

        # LPRO params
        self.eps_low         : float = 0.20   
        self.eps_high        : float = 0.28   
        self.lambda_len      : float = 0.10
        self.eps_r           : float = 1e-8
        self.eps_l           : float = 1e-8

        # training
        self.num_epochs      : int   = 3
        self.lr              : float = 5e-7
        self.weight_decay    : float = 1e-2
        self.warmup_ratio    : float = 0.05
        self.grad_clip       : float = 1.0
        self.grad_accum      : int   = 8      
        self.bf16            : bool  = True

        # logging
        self.log_steps       : int   = 10
        self.save_steps      : int   = 100
        self.seed            : int   = 42

        # huggingface hub
        push_to_hub     : bool  = True
        hf_token        : str   = "" 


def main():
    cfg = Config()
    
    cfg_dict = {k: v for k, v in cfg.__dict__.items() if not k.startswith('__')}
    log.info("Starting training with config:\n" + json.dumps(cfg_dict, indent=2, default=str))
    
    trainer = NexusTrainer(cfg)
    trainer.train()

main()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

modeling_qwen2_rm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen2.5-Math-RM-72B:
- modeling_qwen2_rm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]